# Federated IoT NIDS Feature Engineering Pipeline

This notebook demonstrates a comprehensive feature engineering pipeline for Network Intrusion Detection Systems (NIDS) in IoT environments using federated learning principles. The pipeline is designed for the TRUST_MCNet framework which implements Dynamic Trust-Weighted Aggregation (DTWA) for federated learning.

## Overview

1. **Data Loading**: Loading and exploring multiple IoT datasets for NIDS
2. **Preprocessing**: Cleaning, normalization, and standardization
3. **Feature Engineering**: Comprehensive feature analysis, selection, and transformation
4. **Federated Partitioning**: Creation of non-IID data partitions for federated clients
5. **Model Preparation**: Preparing data for federated model training
6. **Integration with TRUST_MCNet**: Connecting with the federated learning framework

This pipeline focuses on preparing heterogeneous IoT network traffic data for anomaly detection across distributed edge devices.

In [ ]:
# Import necessary libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Any, Optional
from pathlib import Path
import logging
import warnings
import gc

# For preprocessing and feature engineering
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif, VarianceThreshold
from sklearn.decomposition import PCA

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('FeatureEngineering')

# Ignore warnings for cleaner output
warnings.filterwarnings('ignore')

# Set pandas display options for better visibility
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)

# Display versions for reproducibility
print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn: {sklearn.__version__ if 'sklearn' in sys.modules else 'Not imported'}")

# Configure plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set(style="whitegrid", palette="muted", font_scale=1.2)

## 1. IoT Network Dataset Loading and Exploration

In this section, we'll load and explore different IoT Network Intrusion Detection datasets. We'll use the TRUST_MCNet framework's data loading utilities to handle multiple datasets and prepare them for federated learning scenarios.

IoT NIDS datasets often contain features like:
- Network flow statistics (duration, bytes transferred, packet counts)
- Protocol information (TCP, UDP, ICMP)
- Source/destination information
- Connection patterns
- Attack labels (DDoS, scanning, malware, etc.)

We'll explore these datasets to understand their structure and characteristics before preprocessing.

In [ ]:
# Add project root to path for imports
project_root = Path(os.getcwd())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    # Try to import dataset registry from TRUST_MCNet
    from src.trust_mcnet.utils.dataset_registry import DatasetRegistry
    REGISTRY_AVAILABLE = True
except ImportError:
    logger.warning("Could not import DatasetRegistry. Will use direct file loading instead.")
    REGISTRY_AVAILABLE = False

# Function to load IoT datasets
def load_iot_datasets(data_path: str = "data/IoT_Datasets", 
                      dataset_files: Optional[List[str]] = None) -> Dict[str, pd.DataFrame]:
    """
    Load IoT network intrusion datasets from CSV files.
    
    Args:
        data_path: Path to the directory containing dataset files
        dataset_files: List of specific CSV filenames to load (optional)
        
    Returns:
        Dictionary of DataFrames with dataset names as keys
    """
    logger.info(f"Loading IoT datasets from: {data_path}")
    
    # Create the data path if it doesn't exist
    os.makedirs(data_path, exist_ok=True)
    
    # Auto-detect CSV files or use specified files
    if not dataset_files:
        csv_pattern = os.path.join(data_path, "*.csv")
        import glob
        dataset_files = [os.path.basename(f) for f in glob.glob(csv_pattern)]
        logger.info(f"Auto-detected {len(dataset_files)} CSV files")
    
    if not dataset_files:
        logger.warning(f"No CSV files found in {data_path}")
        logger.info("Creating sample datasets for demonstration purposes")
        return _create_sample_datasets(data_path)
    
    # Load each dataset
    datasets = {}
    for file_name in dataset_files:
        try:
            file_path = os.path.join(data_path, file_name)
            dataset_name = file_name.split('.')[0]
            
            logger.info(f"Loading dataset: {dataset_name} from {file_name}")
            df = pd.read_csv(file_path)
            df['dataset_source'] = dataset_name  # Track the source of each sample
            
            datasets[dataset_name] = df
            logger.info(f"Loaded {dataset_name}: {df.shape}")
            
        except Exception as e:
            logger.error(f"Failed to load {file_name}: {e}")
    
    return datasets

def _create_sample_datasets(data_path: str) -> Dict[str, pd.DataFrame]:
    """Create sample datasets for demonstration when real data is not available."""
    datasets = {}
    
    # 1. Edge-IIoT Dataset (Industrial IoT with normal and attack traffic)
    edge_iiot = _create_edge_iiot_sample()
    datasets["Edge_IIoT_Sample"] = edge_iiot
    edge_iiot.to_csv(os.path.join(data_path, "Edge_IIoT_Sample.csv"), index=False)
    
    # 2. MedBIoT Dataset (Medical IoT devices)
    medbiot = _create_medbiot_sample()
    datasets["MedBIoT_Sample"] = medbiot
    medbiot.to_csv(os.path.join(data_path, "MedBIoT_Sample.csv"), index=False)
    
    # 3. ToN-IoT Dataset (IoT and IIoT telemetry with attacks)
    ton_iot = _create_ton_iot_sample()
    datasets["ToN_IoT_Sample"] = ton_iot
    ton_iot.to_csv(os.path.join(data_path, "ToN_IoT_Sample.csv"), index=False)
    
    logger.info(f"Created 3 sample datasets in {data_path}")
    return datasets

def _create_edge_iiot_sample(n_samples: int = 1000):
    """Create synthetic industrial IoT sensor data for testing."""
    np.random.seed(42)
    
    # Common network flow features
    flow_features = {
        'Duration': np.random.exponential(30, n_samples),
        'Protocol': np.random.choice([6, 17, 1], n_samples),  # TCP=6, UDP=17, ICMP=1
        'SrcPort': np.random.randint(1024, 65535, n_samples),
        'DstPort': np.random.randint(1, 65535, n_samples),
        'TotalPkts': np.random.poisson(20, n_samples),
        'TotalBytes': np.random.poisson(2000, n_samples),
        'SrcBytes': np.random.poisson(1000, n_samples),
        'DstBytes': np.random.poisson(1000, n_samples),
        'SrcPkts': np.random.poisson(10, n_samples),
        'DstPkts': np.random.poisson(10, n_samples),
        'SrcIP': np.array(['192.168.1.' + str(np.random.randint(1, 255)) for _ in range(n_samples)]),
        'DstIP': np.array(['10.0.0.' + str(np.random.randint(1, 255)) for _ in range(n_samples)]),
        'TCPFlags': np.random.randint(0, 64, n_samples),
    }
    
    # Industrial IoT specific features
    iiot_features = {
        'SensorID': np.array(['SENSOR_' + str(np.random.randint(1, 100)) for _ in range(n_samples)]),
        'Temperature': np.random.normal(25, 5, n_samples),
        'Pressure': np.random.normal(100, 10, n_samples),
        'Vibration': np.random.normal(0.5, 0.2, n_samples),
        'PowerConsumption': np.random.gamma(5, 2, n_samples),
        'PacketJitter': np.random.exponential(2, n_samples),
        'ConnectionCount': np.random.poisson(3, n_samples),
        'PLC_Read': np.random.poisson(5, n_samples),
        'PLC_Write': np.random.poisson(2, n_samples),
    }
    
    # Create the DataFrame
    df = pd.DataFrame({**flow_features, **iiot_features})
    
    # Generate labels: normal=0, ddos=1, dos=2, scanning=3, other=4
    normal_mask = np.random.random(n_samples) < 0.7  # 70% normal traffic
    
    # Create attack patterns
    df.loc[~normal_mask, 'TotalPkts'] *= 5  # More packets in attacks
    df.loc[~normal_mask, 'TotalBytes'] *= 3  # More bytes in attacks
    df.loc[~normal_mask, 'PacketJitter'] *= 10  # Higher jitter in attacks
    df.loc[~normal_mask, 'ConnectionCount'] += 10  # More connections in attacks
    
    # Create labels
    labels = np.zeros(n_samples)
    attack_indices = np.where(~normal_mask)[0]
    
    # Assign attack types
    ddos_indices = attack_indices[:len(attack_indices)//3]
    dos_indices = attack_indices[len(attack_indices)//3:2*len(attack_indices)//3]
    scan_indices = attack_indices[2*len(attack_indices)//3:]
    
    labels[ddos_indices] = 1  # DDoS
    labels[dos_indices] = 2    # DoS
    labels[scan_indices] = 3   # Scanning
    
    # Add label column
    df['Label'] = np.where(labels == 0, 'Normal', 
                  np.where(labels == 1, 'DDoS', 
                  np.where(labels == 2, 'DoS', 'Scanning')))
    
    df['dataset_source'] = 'Edge_IIoT_Sample'
    
    return df

def _create_medbiot_sample(n_samples: int = 1000):
    """Create synthetic medical IoT device data for testing."""
    np.random.seed(43)
    
    # Common network flow features
    flow_features = {
        'Duration': np.random.exponential(20, n_samples),
        'Protocol': np.random.choice([6, 17], n_samples),  # TCP=6, UDP=17
        'SrcPort': np.random.randint(1024, 65535, n_samples),
        'DstPort': np.random.choice([80, 443, 8080, 23, 5683], n_samples),  # Common IoT ports
        'TotalPkts': np.random.poisson(10, n_samples),
        'TotalBytes': np.random.poisson(1000, n_samples),
        'SrcBytes': np.random.poisson(500, n_samples),
        'DstBytes': np.random.poisson(500, n_samples),
        'SrcIP': np.array(['192.168.2.' + str(np.random.randint(1, 255)) for _ in range(n_samples)]),
        'DstIP': np.array(['203.0.113.' + str(np.random.randint(1, 255)) for _ in range(n_samples)]),
    }
    
    # Medical IoT specific features
    med_features = {
        'DeviceType': np.random.choice(['Monitor', 'Pump', 'Scanner', 'Wearable'], n_samples),
        'HeartRate': np.random.normal(75, 15, n_samples),
        'BloodPressure': np.random.normal(120, 20, n_samples),
        'SpO2': np.random.normal(97, 3, n_samples),
        'Temperature': np.random.normal(37, 1, n_samples),
        'BatteryLevel': np.random.uniform(0, 100, n_samples),
        'SignalStrength': np.random.normal(-65, 15, n_samples),  # dBm
        'DataFrequency': np.random.choice([1, 5, 10, 30, 60], n_samples),  # seconds
        'EncryptionEnabled': np.random.choice([0, 1], n_samples, p=[0.2, 0.8]),
    }
    
    # Create the DataFrame
    df = pd.DataFrame({**flow_features, **med_features})
    
    # Generate labels: normal=0, mirai=1, bashlite=2, ransomware=3
    normal_mask = np.random.random(n_samples) < 0.7  # 70% normal traffic
    
    # Create attack patterns
    df.loc[~normal_mask, 'TotalPkts'] *= 3  # More packets in attacks
    df.loc[~normal_mask, 'TotalBytes'] *= 2  # More bytes in attacks
    df.loc[~normal_mask, 'DataFrequency'] /= 2  # Higher frequency in attacks
    df.loc[~normal_mask, 'SignalStrength'] -= 10  # Lower signal in attacks
    
    # Create labels
    labels = np.zeros(n_samples)
    attack_indices = np.where(~normal_mask)[0]
    
    # Assign attack types
    mirai_indices = attack_indices[:len(attack_indices)//3]
    bashlite_indices = attack_indices[len(attack_indices)//3:2*len(attack_indices)//3]
    ransom_indices = attack_indices[2*len(attack_indices)//3:]
    
    labels[mirai_indices] = 1  # Mirai
    labels[bashlite_indices] = 2  # Bashlite
    labels[ransom_indices] = 3  # Ransomware
    
    # Add label column
    df['Label'] = np.where(labels == 0, 'Normal', 
                  np.where(labels == 1, 'Mirai', 
                  np.where(labels == 2, 'Bashlite', 'Ransomware')))
    
    df['dataset_source'] = 'MedBIoT_Sample'
    
    return df

def _create_ton_iot_sample(n_samples: int = 1000):
    """Create synthetic ToN-IoT network traffic data for testing."""
    np.random.seed(44)
    
    # Common network flow features
    flow_features = {
        'src_ip': np.array(['172.16.' + str(np.random.randint(0, 255)) + '.' + 
                           str(np.random.randint(0, 255)) for _ in range(n_samples)]),
        'dst_ip': np.array(['172.17.' + str(np.random.randint(0, 255)) + '.' + 
                           str(np.random.randint(0, 255)) for _ in range(n_samples)]),
        'src_port': np.random.randint(1024, 65535, n_samples),
        'dst_port': np.random.choice([80, 443, 8080, 22, 23, 25], n_samples),
        'protocol': np.random.choice([6, 17], n_samples),  # TCP=6, UDP=17
        'duration': np.random.exponential(15, n_samples),
        'bytes_in': np.random.poisson(800, n_samples),
        'bytes_out': np.random.poisson(800, n_samples),
        'pkts_in': np.random.poisson(8, n_samples),
        'pkts_out': np.random.poisson(8, n_samples),
    }
    
    # IoT telemetry features
    telemetry_features = {
        'type': np.random.choice(['flow', 'conn', 'dns', 'http'], n_samples),
        'service': np.random.choice(['http', 'dns', 'dhcp', 'mqtt', '-'], n_samples),
        'state': np.random.choice(['established', 'closed', 'ongoing', 'rejected'], n_samples),
        'dur': np.random.exponential(15, n_samples),
        'sbytes': np.random.poisson(800, n_samples),
        'dbytes': np.random.poisson(800, n_samples),
        'sttl': np.random.randint(1, 255, n_samples),
        'dttl': np.random.randint(1, 255, n_samples),
        'sloss': np.random.randint(0, 10, n_samples),
        'dloss': np.random.randint(0, 10, n_samples),
        'sload': np.random.exponential(0.5, n_samples),
        'dload': np.random.exponential(0.5, n_samples),
        'spkts': np.random.poisson(8, n_samples),
        'dpkts': np.random.poisson(8, n_samples),
    }
    
    # Create the DataFrame
    df = pd.DataFrame({**flow_features, **telemetry_features})
    
    # Generate labels: normal=0, injection=1, ddos=2, password=3, scanning=4, xss=5
    normal_mask = np.random.random(n_samples) < 0.7  # 70% normal traffic
    
    # Create attack patterns
    df.loc[~normal_mask, 'pkts_in'] *= 3  # More packets in attacks
    df.loc[~normal_mask, 'bytes_in'] *= 2  # More bytes in attacks
    df.loc[~normal_mask, 'sload'] *= 5  # Higher load in attacks
    df.loc[~normal_mask, 'dload'] *= 5  # Higher load in attacks
    
    # Create labels
    labels = np.zeros(n_samples)
    attack_indices = np.where(~normal_mask)[0]
    
    # Split attack indices into 5 types
    attack_types = 5
    attack_indices_split = np.array_split(attack_indices, attack_types)
    
    # Assign attack types
    for i, indices in enumerate(attack_indices_split):
        labels[indices] = i + 1
    
    # Add label column
    label_map = {
        0: 'Normal',
        1: 'Injection',
        2: 'DDoS',
        3: 'PasswordGuessing',
        4: 'Scanning',
        5: 'XSS'
    }
    df['Label'] = [label_map[int(l)] for l in labels]
    
    df['dataset_source'] = 'ToN_IoT_Sample'
    
    return df

In [ ]:
# Load IoT datasets
datasets = load_iot_datasets()

# Display basic information about each dataset
print(f"Loaded {len(datasets)} datasets:")
for name, df in datasets.items():
    print(f"\n{name}: {df.shape[0]} samples, {df.shape[1]} features")
    print(f"Memory usage: {df.memory_usage().sum() / 1024**2:.2f} MB")
    print(f"Features: {', '.join(list(df.columns)[:5])}...")
    
    # Check for missing values
    missing_values = df.isnull().sum()
    if missing_values.sum() > 0:
        print(f"Missing values: {missing_values.sum()} in {sum(missing_values > 0)} columns")
    else:
        print("No missing values found")
    
    # Show label distribution
    if 'Label' in df.columns:
        label_counts = df['Label'].value_counts()
        print(f"Label distribution:")
        for label, count in label_counts.items():
            print(f"  - {label}: {count} ({count/len(df)*100:.1f}%)")
    
    # Display feature types
    dtypes_count = df.dtypes.value_counts()
    print(f"Feature types:")
    for dtype, count in dtypes_count.items():
        print(f"  - {dtype}: {count}")

# Create a combined dataset for exploration
combined_df = pd.concat(datasets.values(), ignore_index=True)
print(f"\nCombined dataset: {combined_df.shape[0]} samples, {combined_df.shape[1]} features")

In [ ]:
# Visualize key characteristics of the datasets
plt.figure(figsize=(15, 10))

# Plot 1: Class distribution by dataset
plt.subplot(2, 2, 1)
label_counts = []
dataset_names = []

for name, df in datasets.items():
    if 'Label' in df.columns:
        labels = df['Label'].value_counts()
        normal_count = labels.get('Normal', 0)
        attack_count = df.shape[0] - normal_count
        label_counts.append([normal_count, attack_count])
        dataset_names.append(name)

if label_counts:
    label_counts = np.array(label_counts)
    ind = np.arange(len(dataset_names))
    width = 0.35
    
    p1 = plt.bar(ind, label_counts[:, 0], width, label='Normal')
    p2 = plt.bar(ind, label_counts[:, 1], width, bottom=label_counts[:, 0], label='Attack')
    
    plt.ylabel('Sample Count')
    plt.title('Class Distribution by Dataset')
    plt.xticks(ind, dataset_names, rotation=45)
    plt.legend()

# Plot 2: Feature correlation heatmap (using a subset of numeric features)
plt.subplot(2, 2, 2)
if not combined_df.empty:
    numeric_cols = combined_df.select_dtypes(include=[np.number]).columns[:10]
    if len(numeric_cols) > 0:
        sns.heatmap(combined_df[numeric_cols].corr(), annot=False, cmap="coolwarm", vmin=-1, vmax=1)
        plt.title('Feature Correlation (First 10 Numeric Features)')

# Plot 3: Protocol distribution
plt.subplot(2, 2, 3)
protocol_cols = [col for col in combined_df.columns if 'protocol' in col.lower() or 'proto' in col.lower()]
if protocol_cols:
    protocol_col = protocol_cols[0]
    sns.countplot(y=combined_df[protocol_col])
    plt.title(f'Protocol Distribution ({protocol_col})')
    plt.xlabel('Count')
    plt.ylabel('Protocol')
else:
    # If no protocol column found, show duration distribution
    duration_cols = [col for col in combined_df.columns if 'duration' in col.lower() or 'dur' in col.lower()]
    if duration_cols:
        sns.histplot(combined_df[duration_cols[0]].clip(upper=combined_df[duration_cols[0]].quantile(0.95)))
        plt.title(f'Flow Duration Distribution ({duration_cols[0]})')
        plt.xlabel('Duration')
    else:
        plt.text(0.5, 0.5, 'No protocol or duration columns found', ha='center', va='center')

# Plot 4: Bytes distribution by label
plt.subplot(2, 2, 4)
bytes_cols = [col for col in combined_df.columns if 'bytes' in col.lower() or 'byte' in col.lower()]
if bytes_cols and 'Label' in combined_df.columns:
    bytes_col = bytes_cols[0]
    sns.boxplot(x='Label', y=bytes_col, data=combined_df, 
                showfliers=False, order=sorted(combined_df['Label'].unique()))
    plt.title(f'Bytes Distribution by Label ({bytes_col})')
    plt.xticks(rotation=45)
else:
    plt.text(0.5, 0.5, 'No bytes column or Label column found', ha='center', va='center')

plt.tight_layout()
plt.show()

# Feature type distribution
feature_types = combined_df.dtypes.value_counts()
plt.figure(figsize=(8, 5))
feature_types.plot(kind='bar')
plt.title('Feature Type Distribution')
plt.xlabel('Data Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. Comprehensive Data Analysis and Feature Engineering

In this section, we'll perform detailed data analysis and feature engineering on the IoT network traffic data. We'll implement the comprehensive feature engineering process from TRUST_MCNet's `IoTGeneralLoader` to handle IoT-specific characteristics:

1. **Metadata Detection & Removal**: Remove identifiers, timestamps, and IDs
2. **Feature Correlation Analysis**: Find highly correlated feature groups
3. **Feature Variance Analysis**: Remove near-constant features
4. **Mutual Information Selection**: Keep most predictive features
5. **Temporal Feature Extraction**: From timestamp patterns
6. **Protocol-Specific Processing**: Handle protocol-specific features
7. **Feature Standardization**: Scale features appropriately

This approach ensures optimal feature selection for IoT NIDS, balancing between important detection signals while removing redundant information.

In [ ]:
# Define a comprehensive data analysis and feature engineering pipeline
class IoTFeatureEngineer:
    """Comprehensive IoT network traffic feature engineering pipeline."""
    
    def __init__(self, config: Optional[Dict[str, Any]] = None):
        """
        Initialize feature engineering pipeline with configuration.
        
        Args:
            config: Configuration dictionary with preprocessing settings
        """
        self.config = config or {
            'preprocessing': {
                'handle_missing_values': True,
                'missing_value_strategy': 'median',
                'auto_remove_redundant': True,
                'remove_timestamps': True,
                'remove_addresses': True,
                'adaptive_feature_selection': True
            },
            'label_config': {
                'target_column': 'Label',
                'normal_labels': ['Normal', 'Benign', 'BenignTraffic']
            },
            'feature_selection': {
                'remove_metadata': True,
                'remove_timestamps': True,
                'remove_addresses': True,
                'remove_constant': True,
                'remove_high_cardinality': True,
                'high_cardinality_threshold': 0.9,
                'remove_correlated': True,
                'correlation_threshold': 0.9,
                'variance_filtering': True,
                'variance_threshold': 0.01,
                'max_features': None  # No limit by default
            }
        }
        
        self.label_encoders = {}
        self.scaler = None
        self.pca = None
        self.feature_selectors = {}
        
        self.stats = {}
        
    def preprocess_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Enhanced preprocessing with adaptive feature selection for federated learning.
        
        Args:
            df: Input DataFrame with IoT network traffic features
            
        Returns:
            Preprocessed DataFrame ready for model training
        """
        logger.info(f"Starting enhanced preprocessing on dataset with shape: {df.shape}")
        
        # Step 1: Data crawling and analysis
        data_analysis = self._crawl_and_analyze_data(df)
        logger.info(f"Data analysis complete")
        
        # Store key statistics
        self.stats['original_shape'] = df.shape
        self.stats['analysis'] = data_analysis
        
        # Step 2: Handle missing values first
        if self.config['preprocessing'].get('handle_missing_values', True):
            strategy = self.config['preprocessing'].get('missing_value_strategy', 'median')
            df = self._handle_missing_values(df, strategy)
            logger.info(f"After handling missing values: {df.shape}")
        
        # Step 3: Automatic redundant feature removal
        if self.config['preprocessing'].get('auto_remove_redundant', True):
            df = self._remove_redundant_features(df, data_analysis)
            logger.info(f"After removing redundant features: {df.shape}")
        
        # Step 4: Convert non-numeric to numeric
        df = self._convert_categorical_to_numeric(df)
        logger.info(f"After categorical conversion: {df.shape}")
        
        # Step 5: Adaptive feature selection
        if self.config['preprocessing'].get('adaptive_feature_selection', True):
            selected_features = self._adaptive_feature_selection(df, data_analysis)
            
            # Keep only selected features plus target column
            target_col = self.config.get('label_config', {}).get('target_column', 'Label')
            columns_to_keep = selected_features + [target_col] if target_col in df.columns else selected_features
            
            df = df[columns_to_keep]
            logger.info(f"After adaptive feature selection: {df.shape}")
        
        # Step 6: Manual exclusions (if specified)
        exclude_columns = self.config['preprocessing'].get('exclude_columns', [])
        if exclude_columns:
            target_col = self.config.get('label_config', {}).get('target_column', 'Label')
            exclude_columns = [col for col in exclude_columns if col != target_col and col in df.columns]
            if exclude_columns:
                df = df.drop(columns=exclude_columns)
                logger.info(f"After manual exclusions: {df.shape}")
        
        # Store final statistics
        self.stats['processed_shape'] = df.shape
        self.stats['selected_features'] = list(df.columns)
        
        logger.info(f"Final preprocessed dataset shape: {df.shape}")
        return df
    
    def _handle_missing_values(self, df: pd.DataFrame, strategy: str) -> pd.DataFrame:
        """Handle missing values in the dataframe."""
        has_missing = df.isnull().sum().sum() > 0
        
        if not has_missing:
            return df
        
        df_filled = df.copy()
        
        # Separate numeric and non-numeric columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns
        
        # Fill numeric columns based on strategy
        if strategy == 'mean':
            df_filled[numeric_cols] = df_filled[numeric_cols].fillna(df[numeric_cols].mean())
        elif strategy == 'median':
            df_filled[numeric_cols] = df_filled[numeric_cols].fillna(df[numeric_cols].median())
        elif strategy == 'zero':
            df_filled[numeric_cols] = df_filled[numeric_cols].fillna(0)
        else:  # Default is median
            df_filled[numeric_cols] = df_filled[numeric_cols].fillna(df[numeric_cols].median())
        
        # Fill non-numeric with most frequent value
        for col in non_numeric_cols:
            df_filled[col] = df_filled[col].fillna(df[col].mode()[0] if not df[col].mode().empty else "unknown")
        
        return df_filled
    
    def _crawl_and_analyze_data(self, df: pd.DataFrame) -> Dict[str, Any]:
        """Comprehensive data analysis and crawling to understand dataset characteristics."""
        
        logger.info("Starting comprehensive data crawling and analysis...")
        
        analysis = {
            'total_columns': len(df.columns),
            'total_rows': len(df),
            'column_types': {},
            'missing_values': {},
            'unique_value_ratios': {},
            'suspected_metadata': [],
            'suspected_ids': [],
            'suspected_timestamps': [],
            'suspected_addresses': [],
            'numeric_columns': [],
            'categorical_columns': [],
            'constant_columns': [],
            'high_cardinality_columns': [],
            'correlation_groups': [],
            'data_patterns': {},
        }
        
        target_col = self.config.get('label_config', {}).get('target_column', 'Label')
        
        for col in df.columns:
            if col == target_col:
                continue
                
            # Basic type analysis
            dtype = df[col].dtype
            analysis['column_types'][col] = str(dtype)
            
            # Missing value analysis
            missing_ratio = df[col].isnull().sum() / len(df)
            analysis['missing_values'][col] = missing_ratio
            
            # Unique value analysis
            unique_ratio = df[col].nunique() / len(df)
            analysis['unique_value_ratios'][col] = unique_ratio
            
            # Sample values for pattern detection
            sample_values = df[col].dropna().head(20).astype(str)
            
            # Pattern detection
            col_lower = col.lower()
            
            # 1. Metadata detection
            metadata_patterns = ['id', 'uid', 'guid', 'uuid', 'index', 'idx']
            if any(pattern in col_lower for pattern in metadata_patterns):
                analysis['suspected_metadata'].append(col)
            
            # 2. ID detection (high cardinality + patterns)
            if unique_ratio > 0.8 and any(pattern in col_lower for pattern in ['id', 'uid', 'key']):
                analysis['suspected_ids'].append(col)
            
            # 3. Timestamp detection
            timestamp_patterns = ['time', 'date', 'timestamp', 'ts']
            if any(pattern in col_lower for pattern in timestamp_patterns):
                analysis['suspected_timestamps'].append(col)
            
            # Check for timestamp-like content
            if dtype == 'object' and len(sample_values) > 0:
                timestamp_like = sample_values.str.contains(
                    r'\d{4}-\d{2}-\d{2}|\d{2}:\d{2}:\d{2}|\d{10,13}', 
                    regex=True
                ).sum()
                if timestamp_like > len(sample_values) * 0.5:
                    analysis['suspected_timestamps'].append(col)
            
            # 4. Address detection (IP, MAC, host)
            address_patterns = ['host', 'ip', 'addr', 'mac', 'port']
            if any(pattern in col_lower for pattern in address_patterns):
                analysis['suspected_addresses'].append(col)
            
            # Check for IP-like content
            if dtype == 'object' and len(sample_values) > 0:
                ip_like = sample_values.str.contains(
                    r'\d+\.\d+\.\d+\.\d+|[0-9a-fA-F:]+', 
                    regex=True
                ).sum()
                if ip_like > len(sample_values) * 0.3:
                    analysis['suspected_addresses'].append(col)
            
            # 5. Categorize by data type
            if dtype in ['int64', 'float64', 'int32', 'float32']:
                analysis['numeric_columns'].append(col)
                
                # Check if effectively constant
                if df[col].nunique() <= 2:
                    analysis['constant_columns'].append(col)
                    
            else:
                analysis['categorical_columns'].append(col)
                
            # 6. High cardinality detection
            if unique_ratio > 0.9 and len(df) > 50:
                analysis['high_cardinality_columns'].append(col)
            
            # 7. Store data patterns
            if len(sample_values) > 0:
                pattern_info = {
                    'sample_values': sample_values.tolist()[:5],
                    'avg_length': sample_values.str.len().mean() if dtype == 'object' else None,
                    'contains_numbers': sample_values.str.contains(r'\d', regex=True).any() if dtype == 'object' else None,
                    'contains_special_chars': sample_values.str.contains(r'[^a-zA-Z0-9\s]', regex=True).any() if dtype == 'object' else None
                }
                analysis['data_patterns'][col] = pattern_info
        
        # Remove duplicates from lists
        for key in ['suspected_metadata', 'suspected_ids', 'suspected_timestamps', 'suspected_addresses']:
            analysis[key] = list(set(analysis[key]))
        
        # Correlation analysis for numeric columns
        if len(analysis['numeric_columns']) > 1:
            try:
                numeric_df = df[analysis['numeric_columns']].fillna(df[analysis['numeric_columns']].median())
                corr_matrix = numeric_df.corr().abs()
                
                # Find highly correlated groups
                high_corr_threshold = 0.9
                corr_groups = []
                processed_cols = set()
                
                for i, col1 in enumerate(corr_matrix.columns):
                    if col1 in processed_cols:
                        continue
                    
                    corr_group = [col1]
                    for j, col2 in enumerate(corr_matrix.columns):
                        if i != j and col2 not in processed_cols:
                            if corr_matrix.loc[col1, col2] > high_corr_threshold:
                                corr_group.append(col2)
                                processed_cols.add(col2)
                    
                    if len(corr_group) > 1:
                        corr_groups.append(corr_group)
                        processed_cols.update(corr_group)
                
                analysis['correlation_groups'] = corr_groups
            except Exception as e:
                logger.warning(f"Correlation analysis failed: {e}")
        
        return analysis
    
    def _remove_redundant_features(self, df: pd.DataFrame, analysis: Dict[str, Any]) -> pd.DataFrame:
        """Remove clearly redundant features based on analysis."""
        
        target_col = self.config.get('label_config', {}).get('target_column', 'Label')
        original_cols = list(df.columns)
        
        # Features to remove
        to_remove = set()
        
        # Add suspected metadata and ID columns
        to_remove.update(analysis.get('suspected_metadata', []))
        to_remove.update(analysis.get('suspected_ids', []))
        
        # Add timestamp columns if configured
        preprocessing_config = self.config.get('preprocessing', {})
        if preprocessing_config.get('remove_timestamps', True):
            to_remove.update(analysis.get('suspected_timestamps', []))
        
        # Add address columns if configured
        if preprocessing_config.get('remove_addresses', True):
            to_remove.update(analysis.get('suspected_addresses', []))
        
        # Ensure we don't remove the target column
        to_remove.discard(target_col)
        
        # Remove only columns that exist in the dataframe
        to_remove = [col for col in to_remove if col in df.columns]
        
        if to_remove:
            df = df.drop(columns=to_remove)
            logger.info(f"Removed {len(to_remove)} redundant features: {to_remove}")
        
        return df
    
    def _convert_categorical_to_numeric(self, df: pd.DataFrame) -> pd.DataFrame:
        """Convert categorical features to numeric using label encoding."""
        
        categorical_cols = df.select_dtypes(include=['object', 'category']).columns
        
        # Store original DataFrame in case of errors
        df_original = df.copy()
        
        try:
            for col in categorical_cols:
                if col in self.label_encoders:
                    # Use existing encoder
                    encoder = self.label_encoders[col]
                    # Handle unseen values
                    df[col] = df[col].astype(str)
                    unseen = ~df[col].isin(encoder.classes_)
                    if unseen.any():
                        # Add placeholder for unseen values
                        logger.warning(f"Unseen values in {col}, using 'unknown' placeholder")
                        df.loc[unseen, col] = 'unknown'
                    # Transform using existing encoder
                    df[col] = encoder.transform(df[col])
                else:
                    # Create new encoder
                    encoder = LabelEncoder()
                    df[col] = df[col].fillna('unknown').astype(str)
                    df[col] = encoder.fit_transform(df[col])
                    self.label_encoders[col] = encoder
        
        except Exception as e:
            logger.error(f"Error converting categorical to numeric: {e}")
            logger.warning("Falling back to one-hot encoding")
            
            # Fallback to one-hot encoding
            df = df_original
            df = pd.get_dummies(df, columns=list(categorical_cols))
        
        return df
    
    def _adaptive_feature_selection(self, df: pd.DataFrame, analysis: Dict[str, Any]) -> List[str]:
        """Intelligent feature selection based on data analysis results."""
        
        logger.info("Starting adaptive feature selection...")
        
        target_col = self.config.get('label_config', {}).get('target_column', 'Label')
        feature_selection_config = self.config.get('feature_selection', {})
        
        # Get all feature columns (excluding target)
        all_features = [col for col in df.columns if col != target_col]
        selected_features = all_features.copy()
        
        removal_reasons = {}
        
        # 1. Remove suspected metadata and ID columns
        metadata_removal = set(analysis['suspected_metadata'] + analysis['suspected_ids'])
        if metadata_removal and feature_selection_config.get('remove_metadata', True):
            for col in metadata_removal:
                if col in selected_features:
                    selected_features.remove(col)
                    removal_reasons[col] = "Metadata/ID column"
            logger.info(f"Removed {len(metadata_removal)} metadata/ID columns")
        
        # 2. Remove timestamp columns (unless specifically requested)
        if feature_selection_config.get('remove_timestamps', True):
            for col in analysis['suspected_timestamps']:
                if col in selected_features:
                    selected_features.remove(col)
                    removal_reasons[col] = "Timestamp column"
            logger.info(f"Removed {len(analysis['suspected_timestamps'])} timestamp columns")
        
        # 3. Remove address columns (unless specifically requested)
        if feature_selection_config.get('remove_addresses', True):
            for col in analysis['suspected_addresses']:
                if col in selected_features:
                    selected_features.remove(col)
                    removal_reasons[col] = "Address column"
            logger.info(f"Removed {len(analysis['suspected_addresses'])} address columns")
        
        # 4. Remove constant/near-constant columns
        if feature_selection_config.get('remove_constant', True):
            for col in analysis['constant_columns']:
                if col in selected_features:
                    selected_features.remove(col)
                    removal_reasons[col] = "Constant/near-constant"
            logger.info(f"Removed {len(analysis['constant_columns'])} constant columns")
        
        # 5. Remove high cardinality categorical columns
        high_cardinality_threshold = feature_selection_config.get('high_cardinality_threshold', 0.9)
        if feature_selection_config.get('remove_high_cardinality', True):
            high_cardinality_removal = []
            for col in analysis['high_cardinality_columns']:
                if col in selected_features and col in analysis['categorical_columns']:
                    if analysis['unique_value_ratios'][col] > high_cardinality_threshold:
                        selected_features.remove(col)
                        removal_reasons[col] = "High cardinality categorical"
                        high_cardinality_removal.append(col)
            logger.info(f"Removed {len(high_cardinality_removal)} high cardinality categorical columns")
        
        # 6. Handle highly correlated features
        if feature_selection_config.get('remove_correlated', True) and analysis['correlation_groups']:
            corr_removal = []
            for group in analysis['correlation_groups']:
                # Keep only the first feature from each correlation group
                features_in_group = [col for col in group if col in selected_features]
                if len(features_in_group) > 1:
                    to_remove = features_in_group[1:]  # Keep first, remove rest
                    for col in to_remove:
                        selected_features.remove(col)
                        removal_reasons[col] = f"High correlation (group: {features_in_group[0]})"
                        corr_removal.append(col)
            logger.info(f"Removed {len(corr_removal)} highly correlated columns")
        
        # 7. Variance-based filtering for remaining numeric features
        if feature_selection_config.get('variance_filtering', True):
            numeric_features = [col for col in selected_features if col in analysis['numeric_columns']]
            if len(numeric_features) > 0:
                try:
                    variance_threshold = feature_selection_config.get('variance_threshold', 0.01)
                    numeric_df = df[numeric_features].fillna(df[numeric_features].median())
                    
                    # Normalize features for variance calculation
                    scaler = StandardScaler()
                    normalized_data = scaler.fit_transform(numeric_df)
                    
                    selector = VarianceThreshold(threshold=variance_threshold)
                    selector.fit(normalized_data)
                    
                    low_variance_features = [
                        numeric_features[i] for i, keep in enumerate(selector.get_support()) if not keep
                    ]
                    
                    for col in low_variance_features:
                        if col in selected_features:
                            selected_features.remove(col)
                            removal_reasons[col] = "Low variance"
                    
                    logger.info(f"Removed {len(low_variance_features)} low variance numeric columns")
                    
                except Exception as e:
                    logger.warning(f"Variance filtering failed: {e}")
        
        # 8. Mutual information-based selection (if target is available and sklearn is available)
        max_features = feature_selection_config.get('max_features', None)
        if max_features and len(selected_features) > max_features and target_col in df.columns:
            try:
                logger.info(f"Applying mutual information selection to reduce from {len(selected_features)} to {max_features} features")
                
                # Prepare data for mutual information
                X = df[selected_features].copy()
                y = df[target_col].copy()
                
                # Handle missing values
                for col in X.select_dtypes(include=['number']).columns:
                    X[col] = X[col].fillna(X[col].median())
                
                # Encode categorical features
                for col in X.select_dtypes(include=['object', 'category']).columns:
                    X[col] = X[col].fillna('unknown')
                    le = LabelEncoder()
                    X[col] = le.fit_transform(X[col].astype(str))
                
                # Convert label to binary or keep it as is
                if target_col in self.label_encoders:
                    y = self.label_encoders[target_col].transform(y.astype(str))
                elif y.dtype == 'object':
                    le_target = LabelEncoder()
                    y = le_target.fit_transform(y.astype(str))
                    self.label_encoders[target_col] = le_target
                
                # Use classification or regression based on label cardinality
                is_classification = y.nunique() < 20
                if is_classification:
                    mi_scores = mutual_info_classif(X, y, random_state=42)
                else:
                    from sklearn.feature_selection import mutual_info_regression
                    mi_scores = mutual_info_regression(X, y, random_state=42)
                
                # Select top features based on mutual information
                feature_scores = list(zip(selected_features, mi_scores))
                feature_scores.sort(key=lambda x: x[1], reverse=True)
                
                # Keep top features
                top_features = [feat for feat, score in feature_scores[:max_features]]
                removed_features = [feat for feat in selected_features if feat not in top_features]
                
                for col in removed_features:
                    removal_reasons[col] = "Low mutual information"
                
                selected_features = top_features
                logger.info(f"Selected top {len(selected_features)} features based on mutual information")
                
            except Exception as e:
                logger.warning(f"Mutual information selection failed: {e}")
        
        # Log removal summary
        if removal_reasons:
            logger.info("Feature removal summary:")
            reason_counts = {}
            
            for reason in removal_reasons.values():
                reason_counts[reason] = reason_counts.get(reason, 0) + 1
            
            for reason, count in reason_counts.items():
                logger.info(f"  - {reason}: {count} features")
        
        logger.info(f"Feature selection complete: {len(all_features)} -> {len(selected_features)} features")
        
        return selected_features
    
    def prepare_features_labels(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        """Prepare features and labels from dataframe."""
        
        label_config = self.config.get('label_config', {})
        target_column = label_config.get('target_column', 'Label')
        
        if target_column not in df.columns:
            raise ValueError(f"Target column '{target_column}' not found in dataset")
        
        # Separate features and labels
        y = df[target_column].copy()
        X = df.drop(columns=[target_column])
        
        # Convert labels to binary (normal=0, anomaly=1)
        normal_labels = label_config.get('normal_labels', ['Normal'])
        y_binary = (~y.isin(normal_labels)).astype(int)
        
        # Handle categorical features
        categorical_features = X.select_dtypes(include=['object']).columns
        for col in categorical_features:
            # Handle unknown values
            X[col] = X[col].fillna('unknown')
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
        
        # Handle boolean features
        boolean_columns = X.select_dtypes(include=['bool']).columns
        X[boolean_columns] = X[boolean_columns].astype(int)
        
        # Convert to numpy arrays
        X_array = X.values.astype(np.float32)
        y_array = y_binary.values.astype(np.int64)
        
        # Standardize features
        preprocessing_config = self.config.get('preprocessing', {})
        if preprocessing_config.get('standardization', True):
            if self.scaler is None:
                self.scaler = StandardScaler()
                X_array = self.scaler.fit_transform(X_array)
            else:
                X_array = self.scaler.transform(X_array)
        
        logger.info(f"Feature matrix shape: {X_array.shape}")
        logger.info(f"Label distribution: Normal={np.sum(y_array == 0)}, Anomaly={np.sum(y_array == 1)}")
        
        return X_array, y_array

# Create an instance of the feature engineering pipeline
feature_engineer = IoTFeatureEngineer()

# Process each dataset
processed_datasets = {}
for name, df in datasets.items():
    logger.info(f"Processing dataset: {name}")
    processed_df = feature_engineer.preprocess_dataframe(df)
    processed_datasets[name] = processed_df
    
    # Display basic info about processed dataset
    print(f"\nProcessed {name}:")
    print(f"  Original shape: {df.shape} → Processed shape: {processed_df.shape}")
    print(f"  Features: {', '.join(list(processed_df.columns)[:5])}...")
    
    # Display removed features
    removed = set(df.columns) - set(processed_df.columns)
    if removed:
        print(f"  Removed {len(removed)} features, including: {', '.join(list(removed)[:5])}...")
    
    # Display label distribution
    if 'Label' in processed_df.columns:
        label_counts = processed_df['Label'].value_counts()
        normal_count = sum(label_counts.get(label, 0) for label in ['Normal', 'Benign', 'BenignTraffic'])
        attack_count = processed_df.shape[0] - normal_count
        print(f"  Label distribution: Normal={normal_count}, Attack={attack_count}")

# Create a combined processed dataset
combined_processed_df = pd.concat(processed_datasets.values(), ignore_index=True)
print(f"\nCombined processed dataset: {combined_processed_df.shape[0]} samples, {combined_processed_df.shape[1]} features")

In [ ]:
# Visualize the effect of feature engineering
plt.figure(figsize=(16, 12))

# Plot 1: Original vs Processed feature count
plt.subplot(2, 2, 1)
dataset_names = list(datasets.keys())
original_counts = [datasets[name].shape[1] for name in dataset_names]
processed_counts = [processed_datasets[name].shape[1] for name in dataset_names]

x = np.arange(len(dataset_names))
width = 0.35

plt.bar(x - width/2, original_counts, width, label='Original')
plt.bar(x + width/2, processed_counts, width, label='Processed')
plt.xticks(x, dataset_names, rotation=45)
plt.ylabel('Feature Count')
plt.title('Feature Count: Original vs Processed')
plt.legend()

# Plot 2: Feature importance for classification (top 10)
plt.subplot(2, 2, 2)

if 'Label' in combined_processed_df.columns:
    try:
        target_col = 'Label'
        X = combined_processed_df.drop(columns=[target_col])
        y = combined_processed_df[target_col]
        
        # Convert labels to binary (normal=0, anomaly=1)
        normal_labels = ['Normal', 'Benign', 'BenignTraffic']
        y_binary = (~y.isin(normal_labels)).astype(int)
        
        # Handle categorical features
        X_encoded = pd.get_dummies(X)
        
        # Calculate mutual information
        mi_scores = mutual_info_classif(X_encoded, y_binary, random_state=42)
        mi_df = pd.DataFrame({'Feature': X_encoded.columns, 'MI Score': mi_scores})
        mi_df = mi_df.sort_values('MI Score', ascending=False).head(10)
        
        # Plot MI scores
        sns.barplot(x='MI Score', y='Feature', data=mi_df)
        plt.title('Top 10 Features by Mutual Information')
    except Exception as e:
        plt.text(0.5, 0.5, f"Could not calculate feature importance: {e}", ha='center', va='center')
else:
    plt.text(0.5, 0.5, "Label column not found in processed dataset", ha='center', va='center')

# Plot 3: Class separation in 2D feature space (using PCA)
plt.subplot(2, 2, 3)

if 'Label' in combined_processed_df.columns:
    try:
        target_col = 'Label'
        X = combined_processed_df.drop(columns=[target_col])
        y = combined_processed_df[target_col]
        
        # Convert labels to binary (normal=0, anomaly=1)
        normal_labels = ['Normal', 'Benign', 'BenignTraffic']
        y_binary = (~y.isin(normal_labels)).astype(int)
        
        # Handle categorical features
        X_encoded = pd.get_dummies(X)
        
        # Apply PCA
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_encoded)
        
        # Plot PCA results
        plt.scatter(X_pca[y_binary == 0, 0], X_pca[y_binary == 0, 1], alpha=0.5, label='Normal', s=10)
        plt.scatter(X_pca[y_binary == 1, 0], X_pca[y_binary == 1, 1], alpha=0.5, label='Anomaly', s=10)
        plt.title(f'PCA Feature Space (Explained Var: {pca.explained_variance_ratio_.sum():.2f})')
        plt.xlabel('PC1')
        plt.ylabel('PC2')
        plt.legend()
    except Exception as e:
        plt.text(0.5, 0.5, f"Could not perform PCA: {e}", ha='center', va='center')
else:
    plt.text(0.5, 0.5, "Label column not found in processed dataset", ha='center', va='center')

# Plot 4: Feature type distribution before and after processing
plt.subplot(2, 2, 4)
orig_types = combined_df.dtypes.value_counts()
proc_types = combined_processed_df.dtypes.value_counts()

# Combine and plot
all_types = sorted(list(set(orig_types.index) | set(proc_types.index)))
orig_counts = [orig_types.get(t, 0) for t in all_types]
proc_counts = [proc_types.get(t, 0) for t in all_types]

x = np.arange(len(all_types))
width = 0.35

plt.bar(x - width/2, orig_counts, width, label='Original')
plt.bar(x + width/2, proc_counts, width, label='Processed')
plt.xticks(x, [str(t) for t in all_types], rotation=45)
plt.ylabel('Count')
plt.title('Feature Type Distribution: Before vs After')
plt.legend()

plt.tight_layout()
plt.show()

## 5. Federated Data Partitioning for IoT NIDS

In a federated learning scenario, data is distributed across multiple clients (IoT devices or edge nodes). This section demonstrates how to partition the processed datasets to simulate a realistic federated learning environment for TRUST_MCNet.

In [ ]:
class FederatedDataPartitioner:
    """
    Partitions processed datasets for federated learning scenarios, 
    simulating realistic IoT NIDS deployment with heterogeneous data distribution.
    """
    def __init__(self, random_seed=42):
        self.random_seed = random_seed
        np.random.seed(random_seed)
        
    def partition_iid(self, dataset, num_clients, validation_split=0.1, test_split=0.2):
        """
        Partition data in an IID (Independent and Identically Distributed) manner.
        Each client gets a random subset of the data with similar class distribution.
        
        Args:
            dataset (pd.DataFrame): The dataset to partition
            num_clients (int): Number of clients to partition data for
            validation_split (float): Proportion of data for validation
            test_split (float): Proportion of data for testing
            
        Returns:
            dict: Dictionary with client_id -> (X_train, y_train, X_val, y_val, X_test, y_test)
        """
        if 'Label' not in dataset.columns:
            raise ValueError("Dataset must contain a 'Label' column")
            
        # Shuffle dataset
        dataset = dataset.sample(frac=1.0, random_state=self.random_seed)
        
        # Split features and labels
        X = dataset.drop('Label', axis=1)
        y = dataset['Label']
        
        # Convert labels to binary for simplicity (normal=0, anomaly=1)
        normal_labels = ['Normal', 'Benign', 'BenignTraffic']
        y_binary = (~y.isin(normal_labels)).astype(int)
        
        # Create global test set
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y_binary, test_size=test_split, random_state=self.random_seed, stratify=y_binary
        )
        
        # Split data evenly among clients
        client_partitions = {}
        
        # Calculate data points per client
        samples_per_client = len(X_train_val) // num_clients
        
        for i in range(num_clients):
            # Take a slice for this client
            start_idx = i * samples_per_client
            end_idx = (i + 1) * samples_per_client if i < num_clients - 1 else len(X_train_val)
            
            client_X = X_train_val.iloc[start_idx:end_idx].copy()
            client_y = y_train_val.iloc[start_idx:end_idx].copy()
            
            # Split into train and validation
            X_train, X_val, y_train, y_val = train_test_split(
                client_X, client_y, test_size=validation_split, 
                random_state=self.random_seed, stratify=client_y
            )
            
            client_partitions[f"client_{i+1}"] = {
                'train': (X_train, y_train),
                'val': (X_val, y_val),
                'test': (X_test.copy(), y_test.copy())  # All clients share the same test set
            }
            
        return client_partitions

    def partition_non_iid_label_skew(self, dataset, num_clients, validation_split=0.1, test_split=0.2):
        """
        Partition data in a non-IID manner with label distribution skew.
        Different clients get different distributions of classes.
        
        Args:
            dataset (pd.DataFrame): The dataset to partition
            num_clients (int): Number of clients to partition data for
            validation_split (float): Proportion of data for validation
            test_split (float): Proportion of data for testing
            
        Returns:
            dict: Dictionary with client_id -> (X_train, y_train, X_val, y_val, X_test, y_test)
        """
        if 'Label' not in dataset.columns:
            raise ValueError("Dataset must contain a 'Label' column")
            
        # Split features and labels
        X = dataset.drop('Label', axis=1)
        y = dataset['Label']
        
        # Convert labels to binary for simplicity (normal=0, anomaly=1)
        normal_labels = ['Normal', 'Benign', 'BenignTraffic']
        y_binary = (~y.isin(normal_labels)).astype(int)
        
        # Create global test set
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y_binary, test_size=test_split, random_state=self.random_seed
        )
        
        # Split by class label
        df_train_val = pd.concat([X_train_val, pd.Series(y_train_val, index=X_train_val.index, name='Label')], axis=1)
        normal_samples = df_train_val[df_train_val['Label'] == 0]
        attack_samples = df_train_val[df_train_val['Label'] == 1]
        
        # Create different distributions for different clients
        client_partitions = {}
        
        # Normal and attack ratios for each client (creating non-IID distribution)
        # These ratios determine what percentage of normal vs attack samples each client receives
        client_normal_ratios = np.random.dirichlet(alpha=[0.5] * num_clients)
        client_attack_ratios = np.random.dirichlet(alpha=[0.5] * num_clients)
        
        normal_idx = 0
        attack_idx = 0
        
        for i in range(num_clients):
            # Calculate number of samples for this client
            num_normal = int(len(normal_samples) * client_normal_ratios[i])
            num_attack = int(len(attack_samples) * client_attack_ratios[i])
            
            # Get samples for this client
            client_normal = normal_samples.iloc[normal_idx:normal_idx + num_normal]
            client_attack = attack_samples.iloc[attack_idx:attack_idx + num_attack]
            
            # Update indices
            normal_idx += num_normal
            attack_idx += num_attack
            
            # Combine and shuffle client data
            client_data = pd.concat([client_normal, client_attack]).sample(frac=1.0, random_state=self.random_seed)
            client_X = client_data.drop('Label', axis=1)
            client_y = client_data['Label']
            
            # Split into train and validation
            X_train, X_val, y_train, y_val = train_test_split(
                client_X, client_y, test_size=validation_split, 
                random_state=self.random_seed, stratify=client_y if len(set(client_y)) > 1 else None
            )
            
            client_partitions[f"client_{i+1}"] = {
                'train': (X_train, y_train),
                'val': (X_val, y_val),
                'test': (X_test.copy(), y_test.copy())  # All clients share the same test set
            }
            
        return client_partitions
    
    def partition_non_iid_feature_skew(self, dataset, num_clients, validation_split=0.1, test_split=0.2):
        """
        Partition data in a non-IID manner with feature distribution skew.
        Different clients get different distributions of features, simulating
        different IoT device types or networks with varying traffic patterns.
        
        Args:
            dataset (pd.DataFrame): The dataset to partition
            num_clients (int): Number of clients to partition data for
            validation_split (float): Proportion of data for validation
            test_split (float): Proportion of data for testing
            
        Returns:
            dict: Dictionary with client_id -> (X_train, y_train, X_val, y_val, X_test, y_test)
        """
        if 'Label' not in dataset.columns:
            raise ValueError("Dataset must contain a 'Label' column")
            
        # Split features and labels
        X = dataset.drop('Label', axis=1)
        y = dataset['Label']
        
        # Convert labels to binary for simplicity (normal=0, anomaly=1)
        normal_labels = ['Normal', 'Benign', 'BenignTraffic']
        y_binary = (~y.isin(normal_labels)).astype(int)
        
        # Create global test set
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y_binary, test_size=test_split, random_state=self.random_seed, stratify=y_binary
        )
        
        # Group features by correlation or categories
        # For simplicity, we'll use random feature grouping here
        # In a real-world scenario, this would be based on domain knowledge
        features = list(X_train_val.columns)
        np.random.shuffle(features)
        
        # Create feature groups (representing different traffic types)
        num_feature_groups = min(5, len(features))
        feature_groups = np.array_split(features, num_feature_groups)
        
        # Create non-IID distribution across clients
        client_partitions = {}
        
        # Create different distributions for different clients
        df_train_val = pd.concat([X_train_val, pd.Series(y_train_val, index=X_train_val.index, name='Label')], axis=1)
        
        for i in range(num_clients):
            # Each client gets a biased distribution favoring certain feature groups
            # This simulates different device types with different traffic patterns
            
            # Create a custom sampling weight for each row based on feature values
            primary_group = feature_groups[i % num_feature_groups]
            
            # Rows with higher values in the primary feature group get higher weights
            primary_features_mean = X_train_val[primary_group].mean(axis=1)
            weights = scale_to_range(primary_features_mean.values, 0.5, 10)
            
            # Sample based on weights
            sampled_indices = np.random.choice(
                df_train_val.index, 
                size=min(len(df_train_val) // num_clients * 2, len(df_train_val)),
                replace=False,
                p=weights / weights.sum()
            )
            
            client_data = df_train_val.loc[sampled_indices]
            
            # Split into features and labels
            client_X = client_data.drop('Label', axis=1)
            client_y = client_data['Label']
            
            # Split into train and validation
            X_train, X_val, y_train, y_val = train_test_split(
                client_X, client_y, test_size=validation_split, 
                random_state=self.random_seed, stratify=client_y
            )
            
            client_partitions[f"client_{i+1}"] = {
                'train': (X_train, y_train),
                'val': (X_val, y_val),
                'test': (X_test.copy(), y_test.copy())  # All clients share the same test set
            }
            
        return client_partitions

    def get_data_statistics(self, partitioned_data):
        """
        Get statistics about the partitioned data
        
        Args:
            partitioned_data (dict): Output from partition_* methods
            
        Returns:
            dict: Statistics about the partitioned data
        """
        stats = {}
        
        for client_id, data in partitioned_data.items():
            X_train, y_train = data['train']
            X_val, y_val = data['val']
            X_test, y_test = data['test']
            
            # Calculate label distribution
            train_label_dist = pd.Series(y_train).value_counts(normalize=True)
            val_label_dist = pd.Series(y_val).value_counts(normalize=True)
            test_label_dist = pd.Series(y_test).value_counts(normalize=True)
            
            stats[client_id] = {
                'num_train_samples': len(X_train),
                'num_val_samples': len(X_val),
                'num_test_samples': len(X_test),
                'train_label_distribution': train_label_dist.to_dict(),
                'val_label_distribution': val_label_dist.to_dict(),
                'test_label_distribution': test_label_dist.to_dict(),
            }
            
        return stats
    
    def visualize_partitions(self, partitioned_data, title="Data Distribution Across Clients"):
        """
        Visualize the partitioned data distribution
        
        Args:
            partitioned_data (dict): Output from partition_* methods
            title (str): Title for the visualization
        """
        stats = self.get_data_statistics(partitioned_data)
        
        # Plot sample counts and label distributions
        num_clients = len(stats)
        fig, axes = plt.subplots(2, 1, figsize=(12, 10))
        
        # Sample counts
        client_ids = list(stats.keys())
        train_samples = [stats[client_id]['num_train_samples'] for client_id in client_ids]
        val_samples = [stats[client_id]['num_val_samples'] for client_id in client_ids]
        
        x = np.arange(len(client_ids))
        width = 0.35
        
        axes[0].bar(x - width/2, train_samples, width, label='Train')
        axes[0].bar(x + width/2, val_samples, width, label='Validation')
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(client_ids)
        axes[0].set_ylabel('Number of Samples')
        axes[0].set_title('Sample Count Distribution')
        axes[0].legend()
        
        # Label distribution
        label_proportions = np.array([
            [stats[client_id]['train_label_distribution'].get(1, 0) for client_id in client_ids],
            [stats[client_id]['train_label_distribution'].get(0, 0) for client_id in client_ids]
        ])
        
        axes[1].bar(x, label_proportions[0], label='Anomaly', color='red', alpha=0.7)
        axes[1].bar(x, label_proportions[1], bottom=label_proportions[0], 
                   label='Normal', color='green', alpha=0.7)
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(client_ids)
        axes[1].set_ylabel('Proportion')
        axes[1].set_title('Class Distribution (Train Set)')
        axes[1].set_ylim(0, 1.0)
        axes[1].legend()
        
        plt.suptitle(title, fontsize=16)
        plt.tight_layout()
        plt.show()

# Utility function
def scale_to_range(values, min_val, max_val):
    """Scale values to a range between min_val and max_val"""
    if max(values) == min(values):
        return np.ones_like(values)
    return min_val + (max_val - min_val) * (values - min(values)) / (max(values) - min(values))

# Let's partition our processed datasets using different strategies
# For this example, we'll use the combined processed dataset
partitioner = FederatedDataPartitioner(random_seed=42)

# Number of federated clients to simulate
num_clients = 5

# Create different partition distributions
print("Creating IID, Label Skew Non-IID, and Feature Skew Non-IID partitions...")

iid_partitions = partitioner.partition_iid(
    combined_processed_df, 
    num_clients=num_clients
)

non_iid_label_skew_partitions = partitioner.partition_non_iid_label_skew(
    combined_processed_df, 
    num_clients=num_clients
)

non_iid_feature_skew_partitions = partitioner.partition_non_iid_feature_skew(
    combined_processed_df, 
    num_clients=num_clients
)

# Visualize the different partitioning strategies
print("\nData distribution with IID partitioning:")
partitioner.visualize_partitions(iid_partitions, title="IID Partitioning")

print("\nData distribution with Label Skew Non-IID partitioning:")
partitioner.visualize_partitions(non_iid_label_skew_partitions, title="Non-IID Label Skew Partitioning")

print("\nData distribution with Feature Skew Non-IID partitioning:")
partitioner.visualize_partitions(non_iid_feature_skew_partitions, title="Non-IID Feature Skew Partitioning")

## 6. Integration with TRUST_MCNet Federated Framework

Now that we have processed the IoT NIDS datasets and created federated partitions, we'll integrate with the TRUST_MCNet federated learning framework. This includes using the Ray-Flower client implementation with dynamic trust-weighted aggregation.

In [ ]:
# Import TRUST_MCNet components
import sys
import os

# Add the project root to the path if not already there
if os.path.abspath('.') not in sys.path:
    sys.path.append(os.path.abspath('.'))

try:
    # Import TRUST_MCNet components for federated learning
    from src.ray.enhanced_ray_client import EnhancedRayClient
    from src.ray.ray_flwr_client import RayFlowerClient
    from src.trust.trust_evaluator import TrustEvaluator
    from src.trust.quarantine_state import QuarantineState
    
    TRUST_MCNET_AVAILABLE = True
    print("Successfully imported TRUST_MCNet components")
except ImportError as e:
    TRUST_MCNET_AVAILABLE = False
    print(f"Could not import TRUST_MCNet components: {e}")
    print("Will proceed with simulation of TRUST_MCNet integration")

# Define a simple model for IoT NIDS
from sklearn.ensemble import RandomForestClassifier
import joblib

class IoTNIDSModel:
    """
    Simple model wrapper for IoT NIDS classification
    """
    def __init__(self, model_type="random_forest", params=None):
        self.model_type = model_type
        self.params = params or {}
        self.model = None
        
        if model_type == "random_forest":
            self.model = RandomForestClassifier(
                n_estimators=self.params.get("n_estimators", 100),
                max_depth=self.params.get("max_depth", 10),
                random_state=self.params.get("random_state", 42),
            )
    
    def fit(self, X, y):
        """Train the model"""
        self.model.fit(X, y)
        return self
    
    def predict(self, X):
        """Make predictions"""
        return self.model.predict(X)
    
    def predict_proba(self, X):
        """Get prediction probabilities"""
        return self.model.predict_proba(X)
    
    def evaluate(self, X, y):
        """Evaluate the model"""
        from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
        
        y_pred = self.predict(X)
        y_pred_proba = self.predict_proba(X)[:, 1] if len(set(y)) == 2 else None
        
        results = {
            "accuracy": accuracy_score(y, y_pred),
            "precision": precision_score(y, y_pred, average='weighted'),
            "recall": recall_score(y, y_pred, average='weighted'),
            "f1": f1_score(y, y_pred, average='weighted'),
        }
        
        if y_pred_proba is not None:
            results["auc"] = roc_auc_score(y, y_pred_proba)
            
        return results
    
    def save(self, path):
        """Save the model"""
        joblib.dump(self.model, path)
        
    def load(self, path):
        """Load the model"""
        self.model = joblib.load(path)
        return self

# Simulate TRUST_MCNet integration if components are not available
if not TRUST_MCNET_AVAILABLE:
    # Create a simplified version of the TRUST_MCNet client for demonstration
    class SimplifiedTrustMCNetClient:
        """
        Simplified version of TRUST_MCNet client for demonstration purposes
        """
        def __init__(self, client_id, data_partition, model):
            self.client_id = client_id
            self.data_partition = data_partition
            self.model = model
            self.trust_score = 1.0  # Initial trust score
            
        def train(self, num_epochs=1):
            """Train the model on local data"""
            X_train, y_train = self.data_partition['train']
            self.model.fit(X_train, y_train)
            
            # Evaluate on validation set
            X_val, y_val = self.data_partition['val']
            val_results = self.model.evaluate(X_val, y_val)
            
            print(f"Client {self.client_id} - Training completed")
            print(f"Validation results: {val_results}")
            
            return val_results
        
        def evaluate(self):
            """Evaluate the model on test data"""
            X_test, y_test = self.data_partition['test']
            test_results = self.model.evaluate(X_test, y_test)
            
            print(f"Client {self.client_id} - Evaluation results: {test_results}")
            return test_results
        
        def update_trust_score(self, new_score):
            """Update the trust score for this client"""
            old_score = self.trust_score
            self.trust_score = new_score
            print(f"Client {self.client_id} - Trust score updated: {old_score:.4f} -> {new_score:.4f}")

    # Create a simplified version of the TRUST_MCNet server for demonstration
    class SimplifiedTrustMCNetServer:
        """
        Simplified version of TRUST_MCNet server for demonstration purposes
        """
        def __init__(self):
            self.clients = {}
            self.global_model = None
            self.round = 0
            
        def add_client(self, client):
            """Add a client to the server"""
            self.clients[client.client_id] = client
            
        def set_global_model(self, model):
            """Set the global model"""
            self.global_model = model
            
        def federated_averaging(self, weighted=True):
            """
            Simulate federated averaging with trust-weighted aggregation
            
            In a real implementation, this would aggregate model parameters.
            For this demonstration, we'll just average the evaluation metrics.
            """
            self.round += 1
            print(f"\n--- Federated Round {self.round} ---")
            
            # Have clients train on their local data
            for client_id, client in self.clients.items():
                client.train()
            
            # Evaluate clients
            client_metrics = {}
            for client_id, client in self.clients.items():
                client_metrics[client_id] = client.evaluate()
            
            # Update trust scores based on performance (simplified)
            # In the real TRUST_MCNet, this would be based on more complex criteria
            for client_id, metrics in client_metrics.items():
                new_trust_score = min(1.0, metrics['f1'])
                self.clients[client_id].update_trust_score(new_trust_score)
            
            # Calculate weighted average metrics (simulating model aggregation)
            if weighted:
                weights = {client_id: client.trust_score for client_id, client in self.clients.items()}
                total_weight = sum(weights.values())
                weights = {k: v/total_weight for k, v in weights.items()} if total_weight > 0 else {k: 1/len(weights) for k in weights}
            else:
                weights = {client_id: 1/len(self.clients) for client_id in self.clients}
            
            global_metrics = {}
            for metric in ['accuracy', 'precision', 'recall', 'f1']:
                weighted_sum = sum(client_metrics[client_id][metric] * weights[client_id] for client_id in self.clients)
                global_metrics[metric] = weighted_sum
            
            print("\nGlobal Evaluation Results:")
            for metric, value in global_metrics.items():
                print(f"{metric}: {value:.4f}")
            
            print("\nClient Trust Scores:")
            for client_id, client in self.clients.items():
                print(f"{client_id}: {client.trust_score:.4f}")
            
            return global_metrics

# Create federated learning setup with one of our partitioning strategies
# We'll use the non-IID label skew partitioning for this example
print("Setting up federated learning with TRUST_MCNet...")

# Initialize models for each client
client_models = {}

if TRUST_MCNET_AVAILABLE:
    # If TRUST_MCNet components are available, use them
    # This would be implemented with actual TRUST_MCNet components
    print("Using actual TRUST_MCNet components")
    # Implementation would go here
else:
    # Create a simplified simulation of TRUST_MCNet
    print("Using simplified TRUST_MCNet simulation")
    
    # Create server
    server = SimplifiedTrustMCNetServer()
    
    # Create clients with non-IID label skew partitioning
    for client_id, partition in non_iid_label_skew_partitions.items():
        # Create model for this client
        model = IoTNIDSModel()
        
        # Create client
        client = SimplifiedTrustMCNetClient(client_id, partition, model)
        
        # Add client to server
        server.add_client(client)
    
    # Run federated learning for a few rounds
    num_rounds = 3
    print(f"\nRunning {num_rounds} rounds of federated learning with trust-weighted aggregation...")
    
    for i in range(num_rounds):
        server.federated_averaging(weighted=True)
        
    # Compare with traditional federated averaging (equal weights)
    print("\nComparison: Running 1 round with traditional federated averaging (equal weights)...")
    server.federated_averaging(weighted=False)

## 7. Conclusion and Next Steps

In this notebook, we implemented a comprehensive Federated IoT NIDS Feature Engineering Pipeline for TRUST_MCNet. The pipeline includes:

1. **Data Loading and Exploration**: Loaded multiple IoT NIDS datasets and explored their characteristics.
2. **Comprehensive Data Analysis**: Analyzed feature distributions, correlations, and variance.
3. **Advanced Feature Engineering**: Applied preprocessing techniques specific to IoT NIDS data.
4. **Federated Data Partitioning**: Created IID and non-IID (label skew and feature skew) data partitions.
5. **TRUST_MCNet Integration**: Integrated with the TRUST_MCNet federated learning framework for trust-weighted aggregation.

### Next Steps

For production implementation, consider:

1. Implementing custom feature engineering based on specific IoT device types and deployment environments
2. Using more sophisticated anomaly detection models for rare attack types
3. Enhancing the federated learning process with differential privacy to protect sensitive IoT data
4. Implementing adaptive trust evaluation that considers both data quality and client behavior
5. Deploying the model to edge devices with resource constraints using model optimization techniques

The TRUST_MCNet framework provides a robust foundation for addressing security challenges in IoT environments through federated learning with dynamic trust-weighted aggregation.